### Balanced Label Creation
> - cens_dfs == 1 -> event occured (recurrence, death, etc.)
> - cens_dfs == 0 -> censored (no event observed, maybe lost to follow-up or study ended)

We want to focus only on patients with an event occuring (cens_dfs == 1) and compare their DFS time.

In [229]:
!pip install ete3 torch

In [258]:
import os
import re
import random
from collections import defaultdict
from lifelines.utils import concordance_index
import numpy as np
import pandas as pd
from ete3 import Tree
import math
import random
from typing import List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report

In [231]:
import os
!git clone https://github.com/szinja/cell-translation.git
%cd cell-translation/
%ls

Cloning into 'cell-translation'...
remote: Enumerating objects: 977, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 977 (delta 32), reused 50 (delta 14), pack-reused 903 (from 2)
Receiving objects: 100% (977/977), 74.03 MiB | 14.21 MiB/s, done.
Resolving deltas: 100% (327/327), done.
Updating files: 100% (854/854), done.
/content/cell-translation/cell-translation/cell-translation/cell-translation/cell-translation/cell-translation/cell-translation/cell-translation
clinical_data/               merged_patient_data_with_subclonal_scores.csv
cox.ipynb                    patient_embeddings_with_labels.csv
data/                        patient_subclonal_expansion_scores.csv
data_analysis.ipynb          random_forest.ipynb
deepsurv.ipynb               random_forest_predictions.ipynb
deepsurv_model.pt            regression_models.ipynb
embedding_scaler.joblib      tree_and_clinical_data_lstms.ipynb
experiments.ipynb            t

In [232]:
# From TracerX dataset, we want to create a balanced dataset based on DFS time.
tracerx_df = pd.read_csv("data/tracerX.csv")
tracerx_df.columns = tracerx_df.columns.str.strip()
print(tracerx_df.columns)

threshold_dfs_time = 365*3.5 # Set a threshold for DFS time (e.g., 3.5 year)

# Drop rows with missing dfs_time
tracerx_df = tracerx_df.dropna(subset=['dfs_time'])

# Filter datasets
events_df = tracerx_df[tracerx_df['cens_dfs'] == 1]
non_events_df = tracerx_df[(tracerx_df['dfs_time'] > threshold_dfs_time) & (tracerx_df['cens_dfs'] == 0)]

print(f"Number of events: {len(events_df)}, Number of non-events: {len(non_events_df)}")

# Combine and label
combined_events_df = pd.concat([events_df, non_events_df], ignore_index=True)
combined_events_df['shorter_dfs_balanced'] = (combined_events_df['dfs_time'] < threshold_dfs_time).astype(int)

print(combined_events_df['shorter_dfs_balanced'].value_counts())
print(combined_events_df['shorter_dfs_balanced'].value_counts(normalize=True))

Index(['cruk_id', 'tumour_id_muttable_cruk', 'tumour_id_per_patient', 'age',
       'sex', 'ethnicity', 'cigs_perday', 'years_smoking', 'packyears',
       'smoking_status_merged', 'is.family.lung', 'ECOG_PS', 'pathologyTNM',
       'pT_stage_per_patient', 'pN_stage_per_patient', 'LVI_per_patient',
       'PL_per_patient', 'margin_status_per_patient',
       'size_pathology_per_patient', 'Surgery_type', 'histology_lesion1',
       'histology_lesion1_merged', 'lesion1_sampled', 'histology_lesion2',
       'lesion2_sampled', 'histology_multi_full',
       'histology_multi_full_genomically.confirmed', 'LUAD_pred_subtype',
       'adjuvant_treatment_YN', 'adjuvant_treatment_given',
       'num_cycle_na.added', 'CHMPlatDgName_cleaned', 'CHMOthDgName_cleaned',
       'AdjRadStartTime_manual', 'AdjRadEndTime_manual', 'Recurrence_time_use',
       'newPrim_time_use', 'first_dfs_any_event_rec.or.new.primary',
       'first_event_during_followup', 'cens_os', 'os_time', 'cens_dfs',
       'dfs_ti

### Example Tree and clinical string encodings
>- Newick Tree: ((1,11)8,(14,12)9)root; etc.
>- Clinical data: ("age_65 sex_M ECOG_PS_1 smoking_20py"...) etc.

For simplicity, we consider only the primary T1 tumor for every patient.

In [233]:
newick_dir = "trees.txt"  #  file containing patient newick trees
clinical_csv = "data/tracerX.csv"
cruk_id_col = "cruk_id"
newick_file_template = "{cruk_id}.newick"  # e.g., CRUK0361.newick
random_seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 60
lr = 1e-3
embed_dim = 32
mem_dim = 64
# optional: explicit clinical features list (uncomment & set) - otherwise numeric columns auto-detected
clinical_feature_cols = None  # e.g. ['age', 'packyears']

In [234]:
import pandas as pd

# Load mapping
tree_id_map = pd.read_csv("tree_tumourid.csv", index_col=0)
tree_id_map.columns = tree_id_map.columns.str.strip()

# Filter out Tumour2
tree_id_map_filtered = tree_id_map[~tree_id_map['x'].str.contains('Tumour2')].reset_index(drop=True)

# Load Newick trees
with open("trees.txt") as f:
    newick_lines = [line.strip() for line in f if line.strip()]

# Build mapping - filtered IDs only
trees_by_crukid = dict(zip(tree_id_map_filtered['x'], newick_lines))

print(f"Loaded {len(trees_by_crukid)} trees after filtering")

Loaded 392 trees after filtering


In [235]:
removed_ids = sorted(set(tree_id_map['x']) - set(tree_id_map_filtered['x']))
print(f"{len(removed_ids)} IDs removed: {removed_ids}")

9 IDs removed: ['CRUK0030_Tumour2', 'CRUK0223_Tumour2', 'CRUK0372_Tumour2', 'CRUK0555_Tumour2', 'CRUK0586_Tumour2', 'CRUK0620_Tumour2', 'CRUK0704_Tumour2', 'CRUK0721_Tumour2', 'CRUK0881_Tumour2']


### Tree-LSTMs
Parsing a Newick tree string into a Tree-LSTM object.

In [236]:
# pip install ete3 torch
from ete3 import Tree
import torch
import torch.nn as nn
import torch.nn.functional as F

In [237]:
class TreeNode:
    def __init__(self, name=None):
        self.name = name
        self.children = []
        self.idx = None

def parse_newick_to_treenode(newick_str):
    ete_tree = Tree(newick_str, format=1)

    def build_node(ete_node):
        node = TreeNode(name=ete_node.name if ete_node.name else None)
        node.children = [build_node(child) for child in ete_node.children]
        return node

    return build_node(ete_tree)

In [238]:
# Helper function to convert TreeNode to a list of indices
def collect_names(node, names):
    if node.name:
        names.add(node.name)
    for c in node.children:
        collect_names(c, names)

def assign_indices(node, vocab):
    node.idx = vocab.get(node.name, 0)
    for c in node.children:
        assign_indices(c, vocab)

In [239]:
# TreeLSTM dataset class
class ChildSumTreeLSTMCell(nn.Module):
    def __init__(self, in_dim, mem_dim):
        super().__init__()
        self.ioux = nn.Linear(in_dim, 3 * mem_dim)
        self.iouh = nn.Linear(mem_dim, 3 * mem_dim)
        self.fx = nn.Linear(in_dim, mem_dim)
        self.fh = nn.Linear(mem_dim, mem_dim)

    def forward(self, inputs, child_c, child_h):
        if child_h:
            h_sum = torch.sum(torch.stack(child_h, dim=0), dim=0)
        else:
            h_sum = torch.zeros(inputs.size(0), self.iouh.in_features, device=inputs.device)

        iou = self.ioux(inputs) + self.iouh(h_sum)
        i, o, u = torch.chunk(torch.sigmoid(iou), 3, dim=1)
        u = torch.tanh(u)

        f = []
        for h, c in zip(child_h, child_c):
            f_child = torch.sigmoid(self.fx(inputs) + self.fh(h))
            f.append(f_child * c)

        c = i * u + sum(f) if f else i * u
        h = o * torch.tanh(c)
        return c, h

In [240]:
class TreeLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, mem_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.cell = ChildSumTreeLSTMCell(embed_dim, mem_dim)

    def forward(self, tree):
        # Get embeddings for this node
        device = self.emb.weight.device
        if tree.name is None:
            # Placeholder for internal nodes with no explicit label
            idx = torch.tensor([0], dtype=torch.long, device=device)
        else:
            idx = torch.tensor([tree.idx], dtype=torch.long, device=device)

        inputs = self.emb(idx)

        # Recursively compute child states
        child_c, child_h = [], []
        for child in tree.children:
            c, h = self.forward(child)
            child_c.append(c)
            child_h.append(h)

        c, h = self.cell(inputs, child_c, child_h)
        return c, h

In [241]:
# Display first few trees
for cruk_id, newick in list(trees_by_crukid.items())[:5]:
    print(f"{cruk_id}: {newick}")


CRUK0005: (((((((8:24)18:7)12:3,10:10)13:13)9:159,((((17:4)21:89)5:87)19:2)20:178)3:7)4:82,1:203)2;
CRUK0057: ((4:23)2:999,((6:1)5:24)3:1007)1;
CRUK0039: ((4:34)2:958,(5:24)3:971)1;
CRUK0196: (((7:2,8:2)6:12)2:631,(5:4)4:636)1;
CRUK0023: ((((12:11)4:13)13:20)3:195,((8:1e-06)9:2,(10:1e-06)11:3)7:209,2:189)1;


In [242]:
# Building vocabulary from tree names
names = set()
for newick_str in trees_by_crukid.values():
    root = parse_newick_to_treenode(newick_str)
    collect_names(root, names)
vocab = {name: i+1 for i, name in enumerate(sorted(names))}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TreeLSTM(vocab_size=len(vocab)+1, embed_dim=16, mem_dim=32).to(device)
model.eval()

# Generate embeddings for all trees
patient_embeddings = {}
with torch.no_grad():
    for cruk_id, newick_str in trees_by_crukid.items():
        root = parse_newick_to_treenode(newick_str)
        assign_indices(root, vocab)
        _, h = model(root)
        patient_embeddings[cruk_id] = h.squeeze(0).cpu()

# Convert embeddings to DataFrame
df_embeddings = pd.DataFrame.from_dict(
    {pid: emb.numpy() for pid, emb in patient_embeddings.items()},
    orient="index"
).reset_index().rename(columns={"index": "cruk_id"})

In [243]:
# Merge with clinical data
tracerx_df = pd.read_csv("data/tracerX.csv")
tracerx_df.columns = tracerx_df.columns.str.strip()

threshold_dfs_time = 365 * 3.5
events_df = tracerx_df[tracerx_df['cens_dfs'] == 1]
non_events_df = tracerx_df[(tracerx_df['dfs_time'] > threshold_dfs_time) & (tracerx_df['cens_dfs'] == 0)]
combined_df = pd.concat([events_df, non_events_df], ignore_index=True)

combined_df['shorter_dfs_balanced'] = 0
combined_df.loc[(combined_df['dfs_time'] < threshold_dfs_time), 'shorter_dfs_balanced'] = 1

# Merge embeddings with labels
final_df = df_embeddings.merge(
    combined_df[['cruk_id', 'shorter_dfs_balanced']],
    on="cruk_id",
    how="inner"
)

print("Final dataset shape:", final_df.shape)
print(final_df.head())

# Save for downstream modeling
final_df.to_csv("patient_embeddings_with_labels.csv", index=False)

Final dataset shape: (339, 34)
    cruk_id         0         1         2         3         4         5  \
0  CRUK0005  0.225883  0.453543  0.129106  0.221353  0.204677  0.254872   
1  CRUK0057  0.386814  0.294877  0.409495  0.261167  0.296337  0.391342   
2  CRUK0039  0.356496  0.280058  0.408473  0.238867  0.291371  0.376459   
3  CRUK0196  0.429622  0.277807  0.448892  0.256577  0.304669  0.421226   
4  CRUK0023  0.527204  0.392999  0.533826  0.334863  0.400503  0.540411   

          6         7         8  ...        23        24        25        26  \
0  0.267978  0.343711  0.218612  ...  0.360183  0.487037  0.170221  0.172388   
1  0.174597  0.330428  0.154963  ...  0.189164  0.236067  0.259574  0.448599   
2  0.171244  0.314518  0.151755  ...  0.185908  0.228220  0.246486  0.440404   
3  0.194747  0.341830  0.174043  ...  0.204623  0.240325  0.274838  0.440951   
4  0.239140  0.441018  0.215222  ...  0.244427  0.294740  0.419735  0.484278   

         27        28        29      

In [244]:
import pandas as pd

def read_trees_lines(trees_txt_path: str) -> List[str]:
    """Read trees.txt where each line is a Newick string."""
    with open(trees_txt_path, "r") as fh:
        lines = [ln.strip() for ln in fh if ln.strip()]
    return lines

def build_label_vocab_from_mapping(df: pd.DataFrame, trees: List[str], id_col: str = "cruk_id", line_col: str = "line_index"):
    """
    Build vocab by looking up each patient's tree from trees.txt using line mapping.
    - df must have columns [id_col, line_col]
    """
    vocab = NodeLabelVocab()
    for _, row in df.iterrows():
        line_idx = int(row[line_col])
        if line_idx < 0 or line_idx >= len(trees):
            continue
        newick = trees[line_idx]
        try:
            node_names, _ = parse_newick_to_nodes_and_children(newick)
            for nm in node_names:
                vocab.add(nm)
        except Exception as e:
            print(f"Warning: could not parse line {line_idx} for {row[id_col]}: {e}")
            continue
    return vocab

class TreeClinicalDatasetLines(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, trees: List[str],
                 cruk_col="cruk_id", line_col="line_index",
                 time_col="dfs_time", event_col="cens_dfs",
                 clinical_cols=None, vocab=None,
                 node_embedding_dim=16, node_mode="auto"):
        self.df = df.reset_index(drop=True)
        self.trees = trees
        self.cruk_col = cruk_col
        self.line_col = line_col
        self.time_col = time_col
        self.event_col = event_col
        self.clinical_cols = clinical_cols or []
        self.vocab = vocab
        self.node_embedding_dim = node_embedding_dim
        self.node_mode = node_mode

        self.samples = []
        for _, row in self.df.iterrows():
            line_idx = int(row[line_col])
            if line_idx < 0 or line_idx >= len(self.trees):
                continue
            newick = self.trees[line_idx]
            node_names, children = parse_newick_to_nodes_and_children(newick)
            sample = {
                "cruk_id": row[cruk_col],
                "node_names": node_names,
                "children": children,
                "time": float(row[time_col]),
                "event": int(row[event_col]),
                "clinical": {c: row[c] for c in self.clinical_cols} if self.clinical_cols else {}
            }
            self.samples.append(sample)

        if self.clinical_cols:
            clin_vals = self.df[self.clinical_cols].astype(float).fillna(0.0)
            self.clin_mean = clin_vals.mean().values
            self.clin_std = clin_vals.std().replace(0, 1).values
        else:
            self.clin_mean = None
            self.clin_std = None

    def __len__(self):
        return len(self.samples)

    def _node_list_to_tensor(self, node_names):
        # same as before
        return TreeClinicalDataset._node_list_to_tensor(self, node_names)

    def __getitem__(self, idx):
        s = self.samples[idx]
        nodes_tensor, label_idxs = self._node_list_to_tensor(s["node_names"])
        if self.clinical_cols:
            arr = np.array([s["clinical"].get(c, 0.0) for c in self.clinical_cols], dtype=np.float32)
            arr = (arr - self.clin_mean) / self.clin_std
            clin_tensor = torch.tensor(arr, dtype=torch.float32)
        else:
            clin_tensor = torch.tensor([], dtype=torch.float32)
        return nodes_tensor, s["children"], clin_tensor, torch.tensor(s["time"], dtype=torch.float32), torch.tensor(s["event"], dtype=torch.float32)


In [245]:
# Load tree lines and mapping file
trees = read_trees_lines("trees.txt")
tree_map = pd.read_csv("tree_tumourid.csv")

# Rename for clarity
tree_map = tree_map.rename(columns={"x": "cruk_id", "Unnamed: 0": "line_index"})

# Convert to 0-based index
tree_map["line_index"] = tree_map["line_index"] - 1

print(tree_map.head())
print(tree_map.shape)
print(len(trees))

   line_index   cruk_id
0           0  CRUK0005
1           1  CRUK0057
2           2  CRUK0039
3           3  CRUK0196
4           4  CRUK0023
(401, 2)
400


In [246]:
# Merge with your clinical dataframe
tracerx_df = tracerx_df.merge(tree_map, on="cruk_id", how="inner")

# Build vocab
vocab = build_label_vocab_from_mapping(tracerx_df, trees, id_col="cruk_id", line_col="line_index")
print("Built node label vocab size:", len(vocab))

# Build dataset
dataset = TreeClinicalDatasetLines(
    tracerx_df, trees,
    cruk_col="cruk_id", line_col="line_index",
    time_col="dfs_time", event_col="cens_dfs",
    clinical_cols=clinical_feature_cols if clinical_feature_cols else [],
    vocab=vocab, node_embedding_dim=embed_dim, node_mode="label"
)

print(f"Dataset size: {len(dataset)}")

Built node label vocab size: 58
Dataset size: 384


In [257]:
print(len(tracerx_df))
print(tracerx_df.head())
print(trees[:5])  # first 5 lines of your tree list


384
    cruk_id tumour_id_muttable_cruk tumour_id_per_patient  age     sex  \
0  CRUK0034                CRUK0034              CRUK0034   68  Female   
1  CRUK0150                CRUK0150              CRUK0150   81    Male   
2  CRUK0159                CRUK0159              CRUK0159   60    Male   
3  CRUK0090                CRUK0090              CRUK0090   65    Male   
4  CRUK0045                CRUK0045              CRUK0045   85    Male   

        ethnicity  cigs_perday  years_smoking  packyears  \
0    White- Irish         20.0             35     35.000   
1  White- British         44.5             49    109.025   
2  White- British         20.0             38     38.000   
3  White- British         10.0             35     17.500   
4  White- British         10.0             25     12.500   

  smoking_status_merged  ...  cens_dfs  dfs_time cens_dfs_any_event  \
0             Ex-Smoker  ...         0      1849                  0   
1             Ex-Smoker  ...         1      1362

In [263]:
# -----------------------
# Utilities: parse newick -> nodes tensor + children list
# -----------------------
def parse_newick_to_nodes_and_children(newick_text: str) -> Tuple[List[str], List[List[int]]]:
    t = Tree(newick_text, format=1)  # using ete3
    node_list = []
    children = []

    # Map node object -> index
    idx_map = {}
    idx = 0

    # Do a traversal and assign indices
    for n in t.traverse("preorder"):
        idx_map[n] = idx
        # use node name if present, else use empty string
        node_list.append(n.name if n.name is not None else "")
        children.append([])
        idx += 1

    # fill children arrays: ete3 n.children gives node objects
    for n in t.traverse("preorder"):
        parent_idx = idx_map[n]
        for ch in n.children:
            children[parent_idx].append(idx_map[ch])

    # Ensure root is index 0 - etree preorder gives root first so ok
    return node_list, children

# -----------------------
# Node feature encoder
# Two modes:
#   1) numeric parsing: if node names are numeric floats, use direct floats -> feature vector
#   2) label embedding: map string token -> integer id -> nn.Embedding
# We'll build the label vocab from all trees referenced in your clinical dataframe.
# -----------------------
class NodeLabelVocab:
    def __init__(self):
        self.token2idx: dict[str, int] = {}
        # reserve index 0 for unknown / empty token
        self.token2idx[""] = 0
        self.next_idx = 1

    def add(self, token: str):
        if token not in self.token2idx:
            self.token2idx[token] = self.next_idx
            self.next_idx += 1

    def __len__(self):
        return len(self.token2idx)

    def token_to_idx(self, token: str) -> int:
        return self.token2idx.get(token, 0)

# -----------------------
# Build token vocab by scanning Newick files listed in the tracerx dataframe
# Assumes `tracerx_df` contains column with patient IDs `cruk_id` and that newick files follow
# the pattern newick_dir/{cruk_id}.newick or file name template newick_file_template variable.
# If you used a different mapping, adjust the file path logic below accordingly.
# -----------------------
def build_label_vocab_from_df(df: pd.DataFrame, newick_dir: str, cruk_col: str = "cruk_id", file_template: str = "{cruk_id}.newick"):
    vocab = NodeLabelVocab()
    for cruk in df[cruk_col].dropna().unique():
        fpath = os.path.join(newick_dir, file_template.format(cruk_id=cruk))
        if not os.path.exists(fpath):
            continue
        try:
            with open(fpath, "r") as fh:
                newick = fh.read().strip()
            node_names, _ = parse_newick_to_nodes_and_children(newick)
            for nm in node_names:
                vocab.add(nm)
        except Exception as e:
            # skip unreadable
            print("Warning: couldn't read", fpath, ":", e)
            continue
    return vocab

# Build vocab (this may take a moment depending on how many trees you have)
vocab = build_label_vocab_from_df(
    tracerx_df,
    newick_dir=newick_dir,
    cruk_col=cruk_id_col,
    file_template=newick_file_template
)


print("Built node label vocab size:", len(vocab))

# -----------------------
# Dataset that returns (nodes_tensor, children_list, clinical_tensor, time, event)
# Node features: either numeric (if token parseable) OR token embedding indices (int list).
# We'll produce:
#   nodes_tensor: torch.FloatTensor (n_nodes, node_feat_dim)
# For embedding-mode, node_feat_dim will be node_embedding_dim (set below).
# -----------------------
class TreeClinicalDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, newick_dir: str, cruk_col: str = "cruk_id",
                 newick_template: str = "{cruk_id}.newick", time_col: str = "dfs_time",
                 event_col: str = "cens_dfs", clinical_cols: List[str] = None,
                 vocab: NodeLabelVocab = None, node_embedding_dim: int = 16,
                 node_mode: str = "auto"):
        """
        node_mode: "auto" -> try numeric parsing, fallback to "label"
                   "numeric" -> interpret node.name as comma-separated numeric features or single float
                   "label" -> use string label -> integer idx (vocab required)
        """
        self.df = df.reset_index(drop=True)
        self.newick_dir = newick_dir
        self.cruk_col = cruk_col
        self.newick_template = newick_template
        self.time_col = time_col
        self.event_col = event_col
        self.clinical_cols = clinical_cols if clinical_cols is not None else []
        self.vocab = vocab
        self.node_embedding_dim = node_embedding_dim
        self.node_mode = node_mode

        # Preload (path may be missing for some)
        self.samples = []
        for _, row in self.df.iterrows():
            cruk = row[cruk_col]
            fpath = os.path.join(self.newick_dir, self.newick_template.format(cruk_id=cruk))
            if not os.path.exists(fpath):
                # skip or handle missing tree entries
                continue
            with open(fpath, "r") as fh:
                newick = fh.read().strip()
            node_names, children = parse_newick_to_nodes_and_children(newick)
            # store metadata; we will convert node names -> features in __getitem__
            sample = {
                "cruk_id": cruk,
                "node_names": node_names,
                "children": children,
                "time": float(row[self.time_col]),
                "event": int(row[self.event_col])
            }
            if len(self.clinical_cols) > 0:
                sample["clinical"] = row[self.clinical_cols].to_dict()
            else:
                sample["clinical"] = {}
            self.samples.append(sample)

        # Build clinical scaler / column order
        if len(self.clinical_cols) > 0:
            # Simple numeric imputation + scaling: mean/std from dataset (do inside dataset for demo)
            clin_vals = self.df[self.clinical_cols].astype(float).fillna(0.0)
            self.clin_mean = clin_vals.mean().values
            self.clin_std = clin_vals.std().replace(0, 1).values
        else:
            self.clin_mean = None
            self.clin_std = None

    def __len__(self):
        return len(self.samples)

    def _node_list_to_tensor(self, node_names: List[str]):
        """
        Returns:
          - nodes_tensor (n_nodes, feat_dim) as torch.FloatTensor
          - if label-mode, also return token_idx_list for optional nn.Embedding use
        """
        # choose mode
        mode = self.node_mode
        if mode == "auto":
            # if first token parseable float -> numeric mode
            try:
                float(node_names[0])  # may throw
                mode = "numeric"
            except Exception:
                mode = "label"

        if mode == "numeric":
            # parse each node name as single float or comma-separated floats
            arr = []
            for nm in node_names:
                try:
                    if "," in nm:
                        vals = [float(x) for x in nm.split(",")]
                    else:
                        vals = [float(nm)]
                except Exception:
                    # fallback to zero vector
                    vals = [0.0]
                arr.append(vals)
            max_len = max(len(v) for v in arr)
            # pad ragged
            arr_padded = [v + [0.0] * (max_len - len(v)) for v in arr]
            nodes_tensor = torch.tensor(arr_padded, dtype=torch.float32)
            return nodes_tensor, None

        else:  # label mode
            idxs = [self.vocab.token_to_idx(nm) for nm in node_names]
            # We return indices; the model will use nn.Embedding to convert to vectors
            # For clarity return as torch.LongTensor
            return torch.tensor(idxs, dtype=torch.long), idxs

    def __getitem__(self, idx):
        s = self.samples[idx]
        node_names = s["node_names"]
        children = s["children"]
        nodes_tensor, label_idxs = self._node_list_to_tensor(node_names)

        # clinical tensor
        if len(self.clinical_cols) > 0:
            arr = np.array([s["clinical"].get(c, 0.0) for c in self.clinical_cols], dtype=np.float32)
            # normalize
            arr = (arr - self.clin_mean) / self.clin_std
            clin_tensor = torch.tensor(arr, dtype=torch.float32)
        else:
            clin_tensor = torch.tensor([], dtype=torch.float32)
        return nodes_tensor, children, clin_tensor, torch.tensor(s["time"], dtype=torch.float32), torch.tensor(s["event"], dtype=torch.float32)

# -----------------------
# Collate - return lists for trees (we handle per-sample tree processing in model loop)
# -----------------------
def collate_trees_clinical(batch):
    nodes_list = [item[0] for item in batch]
    children_list = [item[1] for item in batch]
    clin_list = [item[2] for item in batch]
    times = torch.stack([item[3] for item in batch])
    events = torch.stack([item[4] for item in batch])
    return nodes_list, children_list, clin_list, times, events

# -----------------------
# TreeSurvivalModel (adapted to accept clinical vector appended to root embedding)
# Uses a learnable embedding for node labels in label mode, or uses numeric node features directly.
# -----------------------
class TreeSurvivalModelWithClinical(nn.Module):
    def __init__(
        self,
        node_mode: str = "label",
        vocab_size: int = None,
        node_emb_dim: int = 16,
        node_feat_dim: int = None,
        tree_hid: int = 64,
        mlp_hidden: int = 64,
        clinical_dim: int = 0,
        dropout: float = 0.2,
    ):
        """
        node_mode: "label" or "numeric"
        - "label": use nn.Embedding(vocab_size, node_emb_dim)
        - "numeric": use numeric node features (node_feat_dim must be set)
        tree_hid: hidden dim of TreeLSTM
        clinical_dim: number of clinical features (can be 0)
        """
        super().__init__()
        self.node_mode = node_mode
        self.clinical_dim = clinical_dim
        self.tree_hid = tree_hid

        if node_mode == "label":
            assert vocab_size is not None, "vocab_size required for label mode"
            self.node_emb = nn.Embedding(vocab_size, node_emb_dim, padding_idx=0)
            lstm_in_dim = node_emb_dim
        else:
            assert node_feat_dim is not None, "node_feat_dim required for numeric mode"
            self.node_emb = None
            lstm_in_dim = node_feat_dim

        # TreeLSTM cell
        self.cell = ChildSumTreeLSTMCell(lstm_in_dim, tree_hid)

        # MLP for risk prediction (root embedding + clinical)
        mlp_in = tree_hid + clinical_dim
        self.mlp = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(mlp_in, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1),
        )

    def forward(self, nodes, children, clinical_tensor=None):
        """
        nodes: LongTensor (label indices) or FloatTensor (numeric features)
        children: adjacency list
        clinical_tensor: FloatTensor (clinical_dim,)
        """
        device = nodes.device
        if self.node_mode == "label":
          nodes_idx = nodes.to(device)             # nodes must be LongTensor
          node_feats = self.node_emb(nodes_idx)    # shape: (n_nodes, node_emb_dim)
        else:
          node_feats = nodes.to(device).float()    # shape: (n_nodes, node_feat_dim)

        # Compute TreeLSTM root embedding
        h_root, c_root = self.compute_tree_states(node_feats, children)
        if h_root.dim() == 1:
            h_root = h_root.unsqueeze(0)

        # Clinical features
        if clinical_tensor is None or clinical_tensor.numel() == 0:
            clin = torch.zeros((h_root.size(0), 0), device=device)
        else:
            if clinical_tensor.dim() == 1:
                clin = clinical_tensor.unsqueeze(0).to(device)
            else:
                clin = clinical_tensor.to(device)

        x = torch.cat([h_root, clin], dim=1)  # (1, tree_hid + clinical_dim)
        risk = self.mlp(x).squeeze(-1)  # scalar
        print("h_root:", h_root.shape, "clin:", clin.shape, "MLP input:", x.shape)
        return risk

    def compute_tree_states(self, node_feats, children):
        """
        Compute TreeLSTM root state using post-order traversal.
        node_feats: (n_nodes, feat_dim)
        children: list of lists (adjacency)
        """
        n_nodes = len(children)
        h, c = [None] * n_nodes, [None] * n_nodes

        def recurse(idx):
            child_h = []
            child_c = []
            for ch_idx in children[idx]:
                recurse(ch_idx)
                child_h.append(h[ch_idx])
                child_c.append(c[ch_idx])
            node_input = node_feats[idx].unsqueeze(0)  # (1, feat_dim)
            c[idx], h[idx] = self.cell(node_input, child_c, child_h)

        recurse(0)  # root at index 0
        return h[0], c[0]

# -----------------------
# Cox loss (same as earlier cell) - copy into notebook if not present
# -----------------------
def cox_ph_loss(risk: torch.Tensor, times: torch.Tensor, events: torch.Tensor):
    # [Use the same stable Breslow implementation you already have in your notebook]
    risk = risk.view(-1)
    times = times.view(-1)
    events = events.view(-1).float()
    order = torch.argsort(times, descending=True)
    risk_sorted = risk[order]
    times_sorted = times[order]
    events_sorted = events[order]
    try:
        denom = torch.logcumsumexp(risk_sorted, dim=0)
    except Exception:
        exp_r = torch.exp(risk_sorted - risk_sorted.max())
        cums = torch.cumsum(exp_r, dim=0)
        denom = torch.log(cums) + risk_sorted.max()
    loss = torch.tensor(0., device=risk.device)
    n = risk_sorted.size(0)
    i = 0
    while i < n:
        t_i = times_sorted[i]
        j = i
        while j < n and times_sorted[j].item() == t_i.item():
            j += 1
        d = events_sorted[i:j].sum()
        if d.item() > 0:
            r_events = risk_sorted[i:j][events_sorted[i:j] == 1].sum()
            denom_block = denom[j-1]
            loss = loss - (r_events - d * denom_block)
        i = j
    n_events = events.sum()
    if n_events > 0:
        loss = loss / n_events
    return loss

# -----------------------
# Concordance index (naive O(N^2) for clarity)
# -----------------------
def concordance_index(pred: np.ndarray, times: np.ndarray, events: np.ndarray):
    # Returns C-index (higher better)
    n = len(pred)
    assert len(times) == n and len(events) == n
    permissible = 0
    concordant = 0
    tied = 0
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if times[i] < times[j] and events[i] == 1:
                permissible += 1
                if pred[i] > pred[j]:
                    concordant += 1
                elif pred[i] == pred[j]:
                    tied += 1
    if permissible == 0:
        return 0.5
    return (concordant + 0.5 * tied) / permissible


# -----------------------
# Toy dataset creation (synthetic)
# -----------------------
def make_random_tree(max_nodes=8, feat_dim=4, p_branch=0.5):
    # Create a random tree: root index 0
    # We'll create nodes incrementally, randomly assign each new node to be child of an existing node
    n_nodes = random.randint(1, max_nodes)
    nodes = []
    children = [[] for _ in range(n_nodes)]
    for i in range(n_nodes):
        # create random node feature
        feat = np.random.randn(feat_dim).astype(np.float32)
        nodes.append(feat)
        if i > 0:
            # attach as child to a random previous node
            parent = random.randint(0, i - 1)
            children[parent].append(i)
    nodes_t = torch.tensor(np.stack(nodes), dtype=torch.float32)
    return nodes_t, children


Built node label vocab size: 1


In [262]:
# -----------------------
# 1. Node mode + clinical columns
# -----------------------
node_mode = "label" if len(vocab) > 1 else "numeric"

clinical_cols = clinical_feature_cols if clinical_feature_cols else []

# -----------------------
# 2. Dataset (line-based)
# -----------------------
dataset = TreeClinicalDatasetLines(
    tracerx_df, trees,
    cruk_col="cruk_id", line_col="line_index",
    time_col="dfs_time", event_col="cens_dfs",
    clinical_cols=clinical_cols,
    vocab=vocab,
    node_embedding_dim=embed_dim,
    node_mode=node_mode
)

print(f"Dataset size: {len(dataset)}")

# -----------------------
# 3. Train/Validation split
# -----------------------
train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    random_state=random_seed
)
print(f"Train samples: {len(train_idx)}, Validation samples: {len(val_idx)}")

train_ds = torch.utils.data.Subset(dataset, train_idx)
val_ds = torch.utils.data.Subset(dataset, val_idx)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True,
                                           collate_fn=collate_trees_clinical)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False,
                                         collate_fn=collate_trees_clinical)

# -----------------------
# 4. Model
# -----------------------
clinical_dim = len(clinical_cols)

model = TreeSurvivalModelWithClinical(
    node_mode=node_mode,
    vocab_size=len(vocab),
    node_emb_dim=embed_dim,
    node_feat_dim=None if node_mode=="label" else embed_dim,
    tree_hid=mem_dim,
    mlp_hidden=64,
    clinical_dim=clinical_dim,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

# -----------------------
# 5. Training + evaluation functions
# -----------------------
def train_epoch_treeclinical(model, optimizer, dataloader, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for nodes_list, children_list, clin_list, times, events in dataloader:
        optimizer.zero_grad()
        risks = []
        for nodes, children, clin in zip(nodes_list, children_list, clin_list):
            if isinstance(nodes, torch.LongTensor) or nodes.dtype == torch.long:
                nodes = nodes.to(device)
            else:
                nodes = nodes.to(device).float()
            clin = clin.to(device) if clin.numel() > 0 else torch.tensor([], device=device)
            risk = model(nodes, children, clin)
            if risk.dim() == 0:
                risk = risk.unsqueeze(0)
            risks.append(risk)
        risks = torch.cat(risks).view(-1).to(device)
        times = times.to(device)
        events = events.to(device)
        loss = cox_ph_loss(risks, times, events)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


def eval_treeclinical(model, dataloader, device):
    model.eval()
    risks_all = []
    times_all = []
    events_all = []
    with torch.no_grad():
        for nodes_list, children_list, clin_list, times, events in dataloader:
            for nodes, children, clin in zip(nodes_list, children_list, clin_list):
                if isinstance(nodes, torch.LongTensor) or getattr(nodes, "dtype", None) == torch.long:
                    nodes = nodes.to(device)
                else:
                    nodes = nodes.to(device).float()
                clin = clin.to(device) if clin.numel() > 0 else torch.tensor([], device=device)
                risk = model(nodes, children, clin)
                if risk.dim() == 0:
                    risk = risk.unsqueeze(0)
                risks_all.extend(risk.cpu().numpy())
                times_all.extend(times.numpy())
                events_all.extend(events.numpy())
    return concordance_index(np.array(risks_all), np.array(times_all), np.array(events_all))

# -----------------------
# 6. Training loop
# -----------------------
for epoch in range(1, epochs + 1):
    train_loss = train_epoch_treeclinical(model, optimizer, train_loader, device)
    val_cidx = eval_treeclinical(model, val_loader, device)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_C-index={val_cidx:.4f}")


Dataset size: 384
Train samples: 307, Validation samples: 77


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x1 and 32x192)

In [ ]:
import copy
from itertools import product

class EarlyStopping:
    """Stop training when validation C-index doesn't improve after patience epochs."""
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = -float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, score, model):
        improved = score > self.best_score + self.min_delta
        if improved:
            self.best_score = score
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


# -----------------------
# Hyperparameter grid
# -----------------------
param_grid = {
    "embed_dim": [16, 32],
    "mem_dim": [64, 128],
    "dropout": [0.1, 0.3],
    "lr": [1e-3, 5e-4],
    "weight_decay": [1e-5, 1e-4],
}

results = []

for embed_dim, mem_dim, dropout, lr, wd in product(
    param_grid["embed_dim"],
    param_grid["mem_dim"],
    param_grid["dropout"],
    param_grid["lr"],
    param_grid["weight_decay"],
):
    print(f"\n=== Training embed={embed_dim}, mem={mem_dim}, dropout={dropout}, lr={lr}, wd={wd} ===")

    dataset = TreeClinicalDataset(
        tracerx_df, newick_dir, cruk_col=cruk_id_col, newick_template=newick_file_template,
        time_col="dfs_time", event_col="cens_dfs",
        clinical_cols=clinical_cols, vocab=vocab,
        node_embedding_dim=embed_dim, node_mode=node_mode
    )
    train_ds = torch.utils.data.Subset(dataset, train_idx)
    val_ds = torch.utils.data.Subset(dataset, val_idx)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_trees_clinical)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_trees_clinical)

    model = TreeSurvivalModelWithClinical(
        node_mode=node_mode,
        vocab_size=len(vocab),
        node_emb_dim=embed_dim,
        tree_hid=mem_dim,
        mlp_hidden=64,
        clinical_dim=len(clinical_cols),
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    early_stopper = EarlyStopping(patience=5)

    best_val = -1
    for epoch in range(1, 50):  # cap at 50, early stopping will cut short
        train_loss = train_epoch_treeclinical(model, optimizer, train_loader, device)
        val_cidx = eval_treeclinical(model, val_loader, device)
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_C-index={val_cidx:.4f}")

        stop = early_stopper.step(val_cidx, model)
        if val_cidx > best_val:
            best_val = val_cidx
        if stop:
            print("Early stopping triggered.")
            break

    early_stopper.restore_best(model)
    results.append(((embed_dim, mem_dim, dropout, lr, wd), best_val))

# Print summary
results.sort(key=lambda x: -x[1])
print("\n=== Hyperparameter tuning results ===")
for params, val in results:
    print(f"embed={params[0]}, mem={params[1]}, dropout={params[2]}, lr={params[3]}, wd={params[4]} -> val_C={val:.4f}")
